# Тренажер: переріз конуса січною площиною

**Мета:** дослідити, як положення січної площини змінює форму перерізу конуса та його проєкції.

Конус задано радіусом основи `R`, висотою `H` і центром основи `(cx, cy, cz)`. Вершина конуса має координати `(cx, cy, cz + H)`.

Січна площина задається рівнянням:

\[
z = cz + a(x-cx) + b(y-cy) + d.
\]

Параметри `a` і `b` задають нахил площини, а `d` — її вертикальне зміщення відносно центра основи.

У тренажері побудовано перетин площини з **бічною поверхнею конуса** та, за наявності, лінію перетину з диском основи. У 3D-вікні червоним показано лінію перерізу.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

def cone_section(cx=0.0, cy=0.0, cz=0.0,
                 R=3.0, H=5.0, a=0.25, b=0.0, d=2.0,
                 show_guides=True):
    # Параметричне задання бічної поверхні:
    # x=cx+R(1-t)cos(phi), y=cy+R(1-t)sin(phi), z=cz+H*t
    phi = np.linspace(0, 2*np.pi, 1200)
    q = R * (a*np.cos(phi) + b*np.sin(phi))
    denom = H + q
    t = np.divide(d + q, denom, out=np.full_like(q, np.nan), where=np.abs(denom) > 1e-10)

    # Залишаємо точки, що належать скінченному конусу: 0 <= t <= 1
    valid = np.isfinite(t) & (t >= -1e-9) & (t <= 1 + 1e-9)
    t_plot = np.where(valid, np.clip(t, 0, 1), np.nan)
    rr = R * (1 - t_plot)
    xs = cx + rr*np.cos(phi)
    ys = cy + rr*np.sin(phi)
    zs = cz + H*t_plot

    # Перетин площини з диском основи z=cz:
    # a(x-cx)+b(y-cy)+d=0. Це пряма, обрізана колом основи.
    base_x, base_y = [], []
    norm2 = a*a + b*b
    if norm2 > 1e-12:
        # Найближча до центра точка прямої у площині основи
        x0 = cx - a*d/norm2
        y0 = cy - b*d/norm2
        dist2 = (a*a + b*b)
        if d*d <= R*R*dist2 + 1e-9:
            half = np.sqrt(max(0.0, R*R - d*d/dist2) / dist2)
            # Напрям уздовж прямої (−b, a)
            s = np.linspace(-half, half, 300)
            base_x = x0 - b*s
            base_y = y0 + a*s
    elif abs(d) < 1e-9:
        # Горизонтальна площина збігається з площиною основи
        base_phi = np.linspace(0, 2*np.pi, 500)
        base_x = cx + R*np.cos(base_phi)
        base_y = cy + R*np.sin(base_phi)

    base_x = np.asarray(base_x)
    base_y = np.asarray(base_y)
    base_z = np.full_like(base_x, cz, dtype=float)

    fig = plt.figure(figsize=(14, 9))
    ax_front = fig.add_subplot(221)
    ax_profile = fig.add_subplot(222)
    ax_top = fig.add_subplot(223)
    ax3d = fig.add_subplot(224, projection="3d")

    # Проєкції конуса
    zline = np.linspace(cz, cz+H, 250)
    rline = R*(1-(zline-cz)/H)
    ax_front.plot(cx-rline, zline, color="steelblue", lw=1.6)
    ax_front.plot(cx+rline, zline, color="steelblue", lw=1.6)
    ax_front.plot([cx-R, cx+R], [cz, cz], color="steelblue", lw=1.6)
    ax_profile.plot(cy-rline, zline, color="steelblue", lw=1.6)
    ax_profile.plot(cy+rline, zline, color="steelblue", lw=1.6)
    ax_profile.plot([cy-R, cy+R], [cz, cz], color="steelblue", lw=1.6)

    # Лінія перерізу бічної поверхні
    ax_front.plot(xs, zs, color="crimson", lw=2.4, label="Переріз")
    ax_profile.plot(ys, zs, color="crimson", lw=2.4)
    ax_top.plot(xs, ys, color="crimson", lw=2.4)

    # Лінія перерізу диска основи
    if base_x.size:
        ax_front.plot(base_x, base_z, color="crimson", lw=2.4)
        ax_profile.plot(base_y, base_z, color="crimson", lw=2.4)
        ax_top.plot(base_x, base_y, color="crimson", lw=2.4)

    if show_guides:
        # Проєкційні зв'язки для кількох характерних точок кривої
        idx = np.linspace(0, len(phi)-1, 9, dtype=int)
        good = idx[valid[idx]]
        if len(good):
            ax_front.scatter(xs[good], zs[good], s=12, color="crimson")
            ax_profile.scatter(ys[good], zs[good], s=12, color="crimson")
            ax_top.scatter(xs[good], ys[good], s=12, color="crimson")
            for i in good[::2]:
                ax_front.plot([xs[i], xs[i]], [cz, zs[i]], color="gray", lw=.6, alpha=.45, ls=":")
                ax_profile.plot([ys[i], ys[i]], [cz, zs[i]], color="gray", lw=.6, alpha=.45, ls=":")

    ax_front.set_title("Фронтальна проєкція (X–Z)")
    ax_front.set_xlabel("X"); ax_front.set_ylabel("Z")
    ax_profile.set_title("Профільна проєкція (Y–Z)")
    ax_profile.set_xlabel("Y"); ax_profile.set_ylabel("Z")
    ax_top.set_title("Горизонтальна проєкція (X–Y)")
    ax_top.set_xlabel("X"); ax_top.set_ylabel("Y")
    for ax in (ax_front, ax_profile, ax_top):
        ax.grid(True, alpha=.25)
        ax.set_aspect("equal", adjustable="datalim")

    # 3D-конусна поверхня
    phi3 = np.linspace(0, 2*np.pi, 100)
    t3 = np.linspace(0, 1, 70)
    PHI, T = np.meshgrid(phi3, t3)
    RR = R*(1-T)
    X = cx + RR*np.cos(PHI)
    Y = cy + RR*np.sin(PHI)
    Z = cz + H*T
    ax3d.plot_surface(X, Y, Z, color="lightsteelblue", alpha=.28, linewidth=0)
    ax3d.plot(cx + R*np.cos(phi3), cy + R*np.sin(phi3), np.full_like(phi3, cz),
              color="steelblue", lw=1.3)

    # Відображення січної площини як обмеженої напівпрозорої поверхні
    xx = np.linspace(cx-R, cx+R, 35)
    yy = np.linspace(cy-R, cy+R, 35)
    XX, YY = np.meshgrid(xx, yy)
    ZZ = cz + a*(XX-cx) + b*(YY-cy) + d
    inside = (XX-cx)**2 + (YY-cy)**2 <= R**2
    ZZ_plot = np.where(inside & (ZZ >= cz-1e-8) & (ZZ <= cz+H+1e-8), ZZ, np.nan)
    ax3d.plot_surface(XX, YY, ZZ_plot, color="orange", alpha=.22, linewidth=0)

    ax3d.plot(xs, ys, zs, color="crimson", lw=3)
    if base_x.size:
        ax3d.plot(base_x, base_y, base_z, color="crimson", lw=3)
    ax3d.set_title("Просторова модель")
    ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
    ax3d.set_box_aspect((1, 1, max(H/R, .4)))
    ax3d.view_init(elev=23, azim=-55)

    fig.suptitle("Переріз конуса січною площиною", fontsize=15)
    plt.tight_layout()
    plt.show()

    if abs(a) < 1e-8 and abs(b) < 1e-8:
        if 0 < d < H:
            print("Тип перерізу: коло (площина паралельна основі).")
        elif abs(d) < 1e-8:
            print("Площина збігається з площиною основи конуса.")
        elif abs(d-H) < 1e-8:
            print("Площина проходить через вершину конуса.")
        else:
            print("Площина не перетинає внутрішню частину конуса або торкається граничної точки.")
    else:
        print("Змінюйте a, b та d, щоб дослідити різні форми перерізу.")
        print("Примітка: лінія на бічній поверхні обчислюється параметрично; червона лінія на основі — перетин із диском основи.")

interact(
    cone_section,
    cx=FloatSlider(min=-3, max=3, step=0.5, value=0, description="cx"),
    cy=FloatSlider(min=-3, max=3, step=0.5, value=0, description="cy"),
    cz=FloatSlider(min=-2, max=2, step=0.5, value=0, description="cz"),
    R=FloatSlider(min=1, max=5, step=0.5, value=3, description="R"),
    H=FloatSlider(min=1, max=7, step=0.5, value=5, description="H"),
    a=FloatSlider(min=-1.5, max=1.5, step=0.05, value=0.25, description="a"),
    b=FloatSlider(min=-1.5, max=1.5, step=0.05, value=0, description="b"),
    d=FloatSlider(min=-1, max=6, step=0.1, value=2, description="d"),
    show_guides=Checkbox(value=True, description="Проєкційні зв'язки")
)


## Практичні завдання

1. **Круговий переріз:** встановіть `a = 0`, `b = 0` і змінюйте `d`. Спостерігайте, як змінюється радіус перерізу.
2. **Нахил уздовж X:** задайте ненульове `a`, а `b = 0`. Порівняйте фронтальну та профільну проєкції.
3. **Нахил уздовж Y:** задайте `a = 0`, а `b` — ненульове.
4. **Похила площина:** змініть одночасно `a`, `b` і `d`. Визначте, де площина перетинає бічну поверхню, а де — основу.
5. **Інженерний опис:** сформулюйте, які параметри задають конус, які — площину, і як із них обчислюються точки перерізу.

**Зверніть увагу:** переріз — це геометричний результат перетину двох заданих об'єктів, а не лінія, намальована довільно. У цій моделі форма конуса задається параметрично, а точки перетину обчислюються з рівняння площини.
